# Customer360 Retail Analytics
## 04_Bronze_Products — Automated Ingestion

**Purpose**
- Incrementally ingest Product CSV files from S3 using Auto Loader.
- Preserve the Bronze source contract exactly.
- Do not perform Silver transformations in Bronze.
- Do not hard-code automation test rows into the production ingestion task.

**Source**
`s3://olist-retail-project/raw/products/`

**Checkpoint**
`s3://olist-retail-project/_checkpoints/products_ingestion/`

**Schema location**
`s3://olist-retail-project/_schemas/products_ingestion/`

**Target**
`workspace.bronze.products`

**Downstream**
`Silver Products` depends on:
- `workspace.bronze.products`
- `workspace.bronze.category_translation`

`Silver Order Items` later depends on:
- `workspace.silver.orders`
- `workspace.silver.products`
- `workspace.silver.sellers`

In [0]:
from pyspark.sql import functions as F

In [0]:
# ================================================================
# STEP 1 — CONFIGURATION
# ================================================================

SOURCE_PATH = "s3://olist-retail-project/raw/products/"

CHECKPOINT_PATH = (
    "s3://olist-retail-project/_checkpoints/products_ingestion/"
)

SCHEMA_LOCATION = (
    "s3://olist-retail-project/_schemas/products_ingestion/"
)

TARGET_TABLE = "workspace.bronze.products"

EXPECTED_COLUMNS = [
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

print("Source       :", SOURCE_PATH)
print("Checkpoint   :", CHECKPOINT_PATH)
print("Schema       :", SCHEMA_LOCATION)
print("Target       :", TARGET_TABLE)

Source       : s3://olist-retail-project/raw/products/
Checkpoint   : s3://olist-retail-project/_checkpoints/products_ingestion/
Schema       : s3://olist-retail-project/_schemas/products_ingestion/
Target       : workspace.bronze.products


In [0]:
# ================================================================
# STEP 2 — DETERMINE FIRST LOAD VS INCREMENTAL LOAD
# ================================================================
#
# If the Bronze table does not exist, this is the initial baseline
# ingestion. Existing raw files must be allowed.
#
# If the Bronze table already exists, only newly arriving files should
# be ingested. Existing files are excluded.
#
# IMPORTANT:
# Do NOT delete/reset the checkpoint on an existing production table.

bronze_exists = spark.catalog.tableExists(TARGET_TABLE)

if bronze_exists:
    bronze_before_df = spark.table(TARGET_TABLE)
    before_count = bronze_before_df.count()

    INCLUDE_EXISTING_FILES = "false"

    print("Load mode              : INCREMENTAL")
    print(f"Current Bronze rows    : {before_count:,}")
    print("includeExistingFiles   : false")

else:
    bronze_before_df = None
    before_count = 0

    INCLUDE_EXISTING_FILES = "true"

    print("Load mode              : INITIAL BASELINE")
    print("Bronze table does not yet exist.")
    print("includeExistingFiles   : true")

Load mode              : INCREMENTAL
Current Bronze rows    : 32,951
includeExistingFiles   : false


In [0]:
# ================================================================
# STEP 3 — VALIDATE EXISTING BRONZE CONTRACT
# ================================================================

if bronze_exists:

    actual_columns = bronze_before_df.columns

    if actual_columns != EXPECTED_COLUMNS:
        raise ValueError(
            "Existing Bronze Products schema does not match the contract.\n"
            f"Expected: {EXPECTED_COLUMNS}\n"
            f"Actual  : {actual_columns}"
        )

    actual_types = dict(bronze_before_df.dtypes)

    non_string_columns = {
        column: actual_types.get(column)
        for column in EXPECTED_COLUMNS
        if actual_types.get(column) != "string"
    }

    if non_string_columns:
        raise ValueError(
            "Existing Bronze Products contains non-string columns.\n"
            f"Expected all STRING types. Actual: {non_string_columns}"
        )

    print("PASS — Existing Bronze Products schema matches the contract.")
    bronze_before_df.printSchema()

PASS — Existing Bronze Products schema matches the contract.
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [0]:
# ================================================================
# STEP 4 — AUTO LOADER STREAM
# ================================================================

products_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option(
            "cloudFiles.includeExistingFiles",
            INCLUDE_EXISTING_FILES
        )
        .option(
            "cloudFiles.schemaLocation",
            SCHEMA_LOCATION
        )
        .option(
            "cloudFiles.inferColumnTypes",
            "false"
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            "none"
        )
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .load(SOURCE_PATH)
        .select(
            F.col("product_id").cast("string"),
            F.col("product_category_name").cast("string"),
            F.col("product_name_lenght").cast("string"),
            F.col("product_description_lenght").cast("string"),
            F.col("product_photos_qty").cast("string"),
            F.col("product_weight_g").cast("string"),
            F.col("product_length_cm").cast("string"),
            F.col("product_height_cm").cast("string"),
            F.col("product_width_cm").cast("string"),
        )
)

if products_stream_df.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Incoming Products schema does not match the Bronze contract.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual  : {products_stream_df.columns}"
    )

print("PASS — Auto Loader stream configured.")
print("PASS — Incoming Products schema matches the Bronze contract.")

PASS — Auto Loader stream configured.
PASS — Incoming Products schema matches the Bronze contract.


In [0]:
# ================================================================
# STEP 5 — INCREMENTAL BRONZE WRITE
# ================================================================

query = (
    products_stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
)

query.awaitTermination()

print("PASS — Products Auto Loader ingestion completed.")

PASS — Products Auto Loader ingestion completed.


In [0]:
# ================================================================
# STEP 6 — POST-INGESTION BRONZE VALIDATION
# ================================================================

bronze_products_df = spark.table(TARGET_TABLE)

after_count = bronze_products_df.count()

print(f"Bronze Products rows after ingestion : {after_count:,}")

if after_count < before_count:
    raise ValueError(
        "Bronze Products population decreased after ingestion."
    )

print("PASS — Bronze Products population is non-decreasing.")

Bronze Products rows after ingestion : 32,951
PASS — Bronze Products population is non-decreasing.


In [0]:
# ================================================================
# STEP 7 — SCHEMA VALIDATION
# ================================================================

if bronze_products_df.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Bronze Products schema changed after ingestion.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual  : {bronze_products_df.columns}"
    )

actual_types_after = dict(bronze_products_df.dtypes)

non_string_columns_after = {
    column: actual_types_after.get(column)
    for column in EXPECTED_COLUMNS
    if actual_types_after.get(column) != "string"
}

if non_string_columns_after:
    raise ValueError(
        "Bronze Products datatype contract failed.\n"
        f"Expected all STRING types. Actual: {non_string_columns_after}"
    )

print("PASS — Bronze Products schema remains unchanged.")
bronze_products_df.printSchema()

PASS — Bronze Products schema remains unchanged.
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [0]:
# ================================================================
# STEP 8 — PRODUCT ID VALIDATION
# ================================================================

null_product_id = bronze_products_df.filter(
    F.col("product_id").isNull()
).count()

blank_product_id = bronze_products_df.filter(
    F.trim(F.col("product_id")) == ""
).count()

print(f"NULL product_id values  : {null_product_id}")
print(f"Blank product_id values : {blank_product_id}")

if null_product_id != 0:
    raise ValueError(
        "Bronze Products quality gate failed: NULL product_id values found."
    )

if blank_product_id != 0:
    raise ValueError(
        "Bronze Products quality gate failed: blank product_id values found."
    )

print("PASS — Bronze product identifiers are populated.")

NULL product_id values  : 0
Blank product_id values : 0
PASS — Bronze product identifiers are populated.


In [0]:
# ================================================================
# STEP 9 — PRODUCT GRAIN VALIDATION
# ================================================================
#
# Product grain = one row per product_id.
# Bronze does not perform deduplication. Therefore duplicates are a
# source-quality failure for this dataset and are not silently removed.

duplicate_product_groups_df = (
    bronze_products_df
        .groupBy("product_id")
        .count()
        .filter(F.col("count") > 1)
)

duplicate_product_groups = duplicate_product_groups_df.count()

print(
    f"Duplicate product_id groups : "
    f"{duplicate_product_groups}"
)

if duplicate_product_groups != 0:
    display(
        duplicate_product_groups_df.orderBy(
            F.desc("count")
        )
    )

    raise ValueError(
        "Bronze Products quality gate failed: "
        "duplicate product_id groups found."
    )

print("PASS — Product business-key grain is unique.")

Duplicate product_id groups : 0
PASS — Product business-key grain is unique.


In [0]:
# ================================================================
# STEP 10 — RAW-PRESERVATION PROFILING
# ================================================================
#
# These fields remain STRING in Bronze.
# NULLs are profiled but are not transformed here.

profile_df = bronze_products_df.select(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(
            F.col("product_category_name").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_category_name"),

    F.sum(
        F.when(
            F.col("product_name_lenght").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_name_lenght"),

    F.sum(
        F.when(
            F.col("product_description_lenght").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_description_lenght"),

    F.sum(
        F.when(
            F.col("product_photos_qty").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_photos_qty"),

    F.sum(
        F.when(
            F.col("product_weight_g").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_weight_g"),

    F.sum(
        F.when(
            F.col("product_length_cm").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_length_cm"),

    F.sum(
        F.when(
            F.col("product_height_cm").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_height_cm"),

    F.sum(
        F.when(
            F.col("product_width_cm").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_width_cm"),
)

display(profile_df)

print("PASS — Bronze Product raw-preservation profiling completed.")

total_rows,null_product_category_name,null_product_name_lenght,null_product_description_lenght,null_product_photos_qty,null_product_weight_g,null_product_length_cm,null_product_height_cm,null_product_width_cm
32951,610,610,610,610,2,2,2,2


PASS — Bronze Product raw-preservation profiling completed.


## IMPORTANT

Automation test validation is intentionally kept OUT of this
production ingestion notebook.

A production ingestion job must not fail simply because there are
no automation test records.

Use the separate validation notebook/cell below after uploading
the test CSV.

In [0]:
# ================================================================
# OPTIONAL TEST VALIDATION — RUN ONLY DURING TESTING
# ================================================================
#
# Uncomment/run this cell only when validating the automation test.
#
# AUTOTEST_PRODUCT_IDS = [
#     "AUTO_TEST_PRODUCT_001",
#     "AUTO_TEST_PRODUCT_002",
#     "AUTO_TEST_PRODUCT_003",
# ]
#
# automation_test_df = (
#     bronze_products_df
#     .filter(
#         F.col("product_id").isin(AUTOTEST_PRODUCT_IDS)
#     )
# )
#
# automation_test_count = automation_test_df.count()
#
# print(
#     f"Bronze automation test rows : "
#     f"{automation_test_count}"
# )
#
# display(
#     automation_test_df
#     .orderBy("product_id")
# )
#
# if automation_test_count != 3:
#     raise ValueError(
#         "Bronze Products automation validation failed. "
#         f"Expected 3 test rows, found {automation_test_count}."
#     )
#
# print(
#     "PASS — All 3 Product automation test records exist in Bronze."
# )

In [0]:
# ================================================================
# FINAL SUMMARY
# ================================================================

print("=" * 72)
print("PRODUCTS AUTOMATED INGESTION — SUCCESS")
print("=" * 72)
print(f"Source       : {SOURCE_PATH}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Schema       : {SCHEMA_LOCATION}")
print(f"Target       : {TARGET_TABLE}")
print(f"Rows before  : {before_count:,}")
print(f"Rows after   : {after_count:,}")

if bronze_exists:
    print("Mode         : Incremental append")
else:
    print("Mode         : Initial baseline + incremental Auto Loader")

print("File handling: Auto Loader")
print("Schema mode  : Strict contract validation")
print("Bronze role  : Raw source preservation")
print("=" * 72)

PRODUCTS AUTOMATED INGESTION — SUCCESS
Source       : s3://olist-retail-project/raw/products/
Checkpoint   : s3://olist-retail-project/_checkpoints/products_ingestion/
Schema       : s3://olist-retail-project/_schemas/products_ingestion/
Target       : workspace.bronze.products
Rows before  : 32,951
Rows after   : 32,951
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict contract validation
Bronze role  : Raw source preservation
